# Unsupervised learning

**Track:** Foundations. **Level:** Foundations. **CPU:** yes.

## Goals

- Cluster a labeled blob dataset and score the clusters against the labels you are not allowed to train on.
- Compress a wide matrix with PCA and check that more components reconstruct it better.
- Embed the same points in two dimensions with UMAP.
- Flag planted outliers with an isolation forest.

## Sources and license

Original notebook following the scikit-learn clustering, PCA, and outlier tutorials (BSD-3) and the UMAP paper and library (BSD-3, McInnes, Healy, and Melville). UMAP is called as a library. Its source is not copied. Labels are used only for scoring.

## Concepts

K-means minimizes squared distance to `k` centers. It does not know the true labels. Adjusted Rand index compares two partitions and is 1 when they match up to a permutation of the names. PCA finds orthogonal directions of variance. The reconstruction from `k` components is the best rank-`k` approximation in squared error. UMAP places neighbors near each other in a lower dimension. It is a visualization tool here, not a classifier. Isolation Forest scores how easy a point is to separate with random cuts. Easy-to-isolate points are the candidates for anomalies.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'umap-learn'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score

from labs._shared.seed import seed_everything

seed_everything(0)
rng = np.random.default_rng(0)


## Guided exercise: k-means and PCA

The blobs are generated with three centers. K-means is not given that fact as a feature. It is given `n_clusters=3` as a choice you made. PCA is fit on a 10-dimensional matrix whose first three coordinates were stretched, so a few components should explain most of the variance.


In [ ]:
Xb, yb = make_blobs(n_samples=400, centers=3, n_features=2, cluster_std=0.55, random_state=0)
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Xb)
ari = adjusted_rand_score(yb, labels)
print("adjusted Rand", round(ari, 3))
assert ari > 0.9

fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(Xb[:, 0], Xb[:, 1], c=labels, s=12, cmap="tab10")
ax.set_title(f"k-means, ARI {ari:.2f}")
plt.tight_layout()
plt.show()

Xw = rng.normal(size=(250, 10))
Xw[:, :3] *= 4.0
pca = PCA(random_state=0).fit(Xw)
print("explained variance ratio", np.round(pca.explained_variance_ratio_, 3))
assert np.isclose(pca.explained_variance_ratio_.sum(), 1.0)
assert pca.explained_variance_ratio_[:3].sum() > 0.8

def reconstruction_mse(components: int) -> float:
    model = PCA(n_components=components, random_state=0).fit(Xw)
    restored = model.inverse_transform(model.transform(Xw))
    return float(np.mean((Xw - restored) ** 2))

mse2, mse6 = reconstruction_mse(2), reconstruction_mse(6)
print("reconstruction MSE k=2", round(mse2, 3), "k=6", round(mse6, 3))
assert mse6 < mse2


## Guided exercise: UMAP and outliers

UMAP is fit with `compatibility_layout=True` so the embedding matches the library's previous layout algorithm. The assertion is the output shape and a finite embedding, not a particular picture. The outlier set is 300 inliers from a standard normal and 25 points shifted far away. Precision is the fraction of isolation-forest flags that are real planted outliers.


In [ ]:
embedding = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1, random_state=0, compatibility_layout=True,
).fit_transform(Xb)
print("UMAP shape", embedding.shape)
assert embedding.shape == (len(Xb), 2)
assert np.isfinite(embedding).all()

fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(embedding[:, 0], embedding[:, 1], c=yb, s=12, cmap="tab10")
ax.set_title("UMAP of the blobs")
plt.tight_layout()
plt.show()

inliers = rng.normal(size=(300, 6))
outliers = rng.normal(loc=6.0, size=(25, 6))
Xo = np.vstack([inliers, outliers])
truth = np.array([0] * len(inliers) + [1] * len(outliers))
flags = IsolationForest(n_estimators=200, contamination=0.08, random_state=0).fit_predict(Xo) == -1
precision = float(truth[flags].mean()) if flags.any() else 0.0
recall = float(flags[truth == 1].mean())
print(f"outlier precision {precision:.2f}  recall {recall:.2f}  flags {int(flags.sum())}")
assert precision > 0.7
print("LAB OK")


## Your turn

Plot k-means inertia for `k` from 1 to 6 on the blob data. Inertia must fall as `k` rises. That plot does not tell you the true number of clusters by itself. The labels did.

## Solution


In [ ]:
inertias = []
for k in range(1, 7):
    inertias.append(KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xb).inertia_)
print(pd.DataFrame({"k": range(1, 7), "inertia": inertias}).round(1).to_string(index=False))
assert all(a > b for a, b in zip(inertias, inertias[1:]))


## Beyond the course

Refit UMAP at `n_neighbors` 5 and 50 and report the mean distance from each point to its nearest other point in the embedding. A smaller neighborhood parameter is allowed to fragment the picture. The measurement is the comparison. It is not a claim that one picture is the data.


In [ ]:
def mean_nearest(z):
    d = np.sqrt(((z[:, None, :] - z[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(d, np.inf)
    return float(d.min(axis=1).mean())

for neighbors in (5, 50):
    z = umap.UMAP(
        n_components=2, n_neighbors=neighbors, min_dist=0.1, random_state=0, compatibility_layout=True,
    ).fit_transform(Xb)
    print(f"n_neighbors {neighbors} mean nearest-neighbor distance {mean_nearest(z):.3f}")


## Research questions

1. Adjusted Rand used the blob labels. What can you monitor in production when you do not have them?
2. PCA reconstruction error fell with more components. When is a lower reconstruction error a worse representation for a downstream classifier?
3. The isolation forest was told a contamination rate close to the true outlier rate. What happens to precision if you pass 0.01 or 0.3 instead?

## Further reading

- scikit-learn user guide, clustering, PCA, and novelty/outlier detection (BSD-3).
- McInnes, Healy, and Melville, *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction* (2018). Library license BSD-3.
- Liu, Ting, and Zhou, *Isolation Forest* (ICDM 2008).
